[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/study_notes/week05_cuped.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 5: CUPED — Controlled-experiment Using Pre-Experiment Data

**Course**: Advanced A/B Testing | **Context**: QuickCart (online electronics retailer)

**Your role**: Data scientist on the experimentation platform team. You need to detect smaller effects with the same sample size — or equivalently, run shorter experiments. CUPED is the industry-standard technique for variance reduction that makes this possible.

---

## What You’ll Learn

1. The key idea: use pre-experiment data as a covariate to reduce metric variance
2. The CUPED formula and derivation of the optimal theta
3. Hands-on implementation with synthetic QuickCart data
4. CUPED + ML: using model predictions as the covariate
5. Practical considerations and pitfalls
6. Quantifying the variance reduction you can expect

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict

try:
    import lightgbm as lgb
    HAS_LGBM = True
except ImportError:
    !pip install lightgbm -q
    import lightgbm as lgb
    HAS_LGBM = True

from collections import defaultdict

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---

## 1. The Core Idea

### The Problem: High Variance Kills Sensitivity

In an A/B test, the minimum detectable effect (MDE) is proportional to the standard deviation of your metric:

$$
\text{MDE} \propto \frac{\sigma}{\sqrt{n}}
$$

If your metric has high variance, you need enormous sample sizes (or very long experiments) to detect small effects. At QuickCart, purchase totals vary wildly — some users buy nothing, others drop thousands on a new laptop.

### The Insight: Pre-Experiment Behavior is Predictive

A user who spent a lot last month will likely spend a lot this month, regardless of which experimental group they’re in. This predictable component of the metric is **noise** from the experiment’s perspective — it adds variance without adding information about the treatment effect.

**CUPED removes this predictable variance** by subtracting it out, using data collected *before* the experiment started.

### The Formula

Given:
- $Y$ = metric measured during the experiment (e.g., purchase total this month)
- $X$ = covariate measured before the experiment (e.g., purchase total last month)

The CUPED-adjusted metric is:

$$
Y_{\text{cuped}} = Y - \theta \cdot (X - \bar{X})
$$

where the optimal coefficient is:

$$
\theta^* = \frac{\text{Cov}(Y, X)}{\text{Var}(X)}
$$

This is the same $\theta$ as the slope in a simple linear regression of $Y$ on $X$.

### Why This Doesn’t Change the Treatment Effect

Since $X$ is measured **before** randomization, $\mathbb{E}[X]$ is the same in both treatment and control. Therefore:

$$
\mathbb{E}[Y_{\text{cuped}}^{\text{treatment}}] - \mathbb{E}[Y_{\text{cuped}}^{\text{control}}] = \mathbb{E}[Y^{\text{treatment}}] - \mathbb{E}[Y^{\text{control}}]
$$

The mean difference (the treatment effect) is preserved. Only the variance changes.

:::{admonition} Full derivation of optimal theta (click to expand)
:class: dropdown

We want to find $\theta$ that minimizes $\text{Var}(Y_{\text{cuped}})$.

$$
\text{Var}(Y_{\text{cuped}}) = \text{Var}(Y - \theta X)
= \text{Var}(Y) + \theta^2 \text{Var}(X) - 2\theta \text{Cov}(Y, X)
$$

Take the derivative with respect to $\theta$ and set to zero:

$$
\frac{d}{d\theta} \text{Var}(Y_{\text{cuped}}) = 2\theta \text{Var}(X) - 2\text{Cov}(Y, X) = 0
$$

Solving:

$$
\theta^* = \frac{\text{Cov}(Y, X)}{\text{Var}(X)}
$$

Substituting back, the minimum variance is:

$$
\text{Var}(Y_{\text{cuped}}) = \text{Var}(Y)\left(1 - \rho^2_{XY}\right)
$$

where $\rho_{XY}$ is the Pearson correlation between $X$ and $Y$.

**Key result**: The variance reduction factor is $(1 - \rho^2)$. If the correlation between pre-period and pilot-period metric is 0.7, you reduce variance by $1 - 0.49 = 51\%$. This is equivalent to doubling your sample size!
:::

---

## 2. Demonstration: Session Duration Example

Let’s start with a simple synthetic example. QuickCart is testing a new homepage layout and measuring average session duration.

In [ ]:
np.random.seed(42)
n_users = 5000

# Each user has a baseline engagement level (unobserved)
user_baseline = np.random.exponential(scale=10, size=n_users)

# Pre-experiment period: session duration = baseline + noise
pre_experiment = user_baseline + np.random.normal(0, 3, size=n_users)
pre_experiment = np.maximum(pre_experiment, 0.5)  # minimum 0.5 min

# Experiment period: session duration = baseline + noise + treatment effect for half
noise_experiment = np.random.normal(0, 3, size=n_users)
treatment_effect = np.zeros(n_users)
treatment_mask = np.arange(n_users) >= n_users // 2
treatment_effect[treatment_mask] = 0.8  # treatment adds 0.8 minutes

experiment_metric = user_baseline + noise_experiment + treatment_effect
experiment_metric = np.maximum(experiment_metric, 0.5)

# Split into control and treatment
Y_control = experiment_metric[~treatment_mask]
Y_treatment = experiment_metric[treatment_mask]
X_control = pre_experiment[~treatment_mask]
X_treatment = pre_experiment[treatment_mask]

print(f"True treatment effect: 0.8 minutes")
print(f"Observed difference (Y_treatment - Y_control): {Y_treatment.mean() - Y_control.mean():.3f}")
print(f"Correlation between pre and experiment period: {np.corrcoef(pre_experiment, experiment_metric)[0,1]:.3f}")

In [ ]:
# Apply CUPED
# Compute theta using all data (control + treatment pooled for covariate stats)
X_all = np.concatenate([X_control, X_treatment])
Y_all = np.concatenate([Y_control, Y_treatment])

theta = np.cov(Y_all, X_all)[0, 1] / np.var(X_all, ddof=1)
print(f"Optimal theta: {theta:.4f}")

# CUPED-adjusted metrics
X_mean = X_all.mean()
Y_cuped_control = Y_control - theta * (X_control - X_mean)
Y_cuped_treatment = Y_treatment - theta * (X_treatment - X_mean)

# Compare variance
var_original = (np.var(Y_control, ddof=1) + np.var(Y_treatment, ddof=1)) / 2
var_cuped = (np.var(Y_cuped_control, ddof=1) + np.var(Y_cuped_treatment, ddof=1)) / 2

print(f"\nVariance comparison:")
print(f"  Original metric variance:  {var_original:.3f}")
print(f"  CUPED metric variance:     {var_cuped:.3f}")
print(f"  Variance reduction:        {(1 - var_cuped/var_original)*100:.1f}%")

# T-test comparison
_, p_original = stats.ttest_ind(Y_treatment, Y_control)
_, p_cuped = stats.ttest_ind(Y_cuped_treatment, Y_cuped_control)

print(f"\nStatistical test results:")
print(f"  Original p-value: {p_original:.4f}")
print(f"  CUPED p-value:    {p_cuped:.6f}")
print(f"\n  CUPED effect estimate: {Y_cuped_treatment.mean() - Y_cuped_control.mean():.3f} (true: 0.800)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Original metric distributions
axes[0].hist(Y_control, bins=50, alpha=0.5, label='Control', density=True)
axes[0].hist(Y_treatment, bins=50, alpha=0.5, label='Treatment', density=True)
axes[0].axvline(Y_control.mean(), color='blue', linestyle='--', linewidth=2)
axes[0].axvline(Y_treatment.mean(), color='orange', linestyle='--', linewidth=2)
axes[0].set_title(f'Original Metric (Var={var_original:.1f})')
axes[0].set_xlabel('Session Duration (min)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: CUPED-adjusted metric distributions
axes[1].hist(Y_cuped_control, bins=50, alpha=0.5, label='Control (CUPED)', density=True)
axes[1].hist(Y_cuped_treatment, bins=50, alpha=0.5, label='Treatment (CUPED)', density=True)
axes[1].axvline(Y_cuped_control.mean(), color='blue', linestyle='--', linewidth=2)
axes[1].axvline(Y_cuped_treatment.mean(), color='orange', linestyle='--', linewidth=2)
axes[1].set_title(f'CUPED-Adjusted Metric (Var={var_cuped:.1f})')
axes[1].set_xlabel('CUPED Session Duration (min)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice: distributions are tighter after CUPED, making the difference more visible.")

---

## 3. Variance Reduction vs Correlation

The theoretical variance reduction is $(1 - \rho^2)$. Let’s verify this empirically and understand what it means for experiment design.

In [ ]:
correlations = np.linspace(0.0, 0.99, 20)
empirical_reductions = []
theoretical_reductions = []

for rho in correlations:
    # Generate correlated pre/post data
    n = 10000
    # Use Cholesky decomposition to create correlated normals
    mean = [0, 0]
    cov_matrix = [[1, rho], [rho, 1]]
    data = np.random.multivariate_normal(mean, cov_matrix, size=n)
    X = data[:, 0]  # pre-period
    Y = data[:, 1]  # experiment period
    
    # Apply CUPED
    theta = np.cov(Y, X)[0, 1] / np.var(X, ddof=1)
    Y_cuped = Y - theta * (X - X.mean())
    
    empirical_reductions.append(1 - np.var(Y_cuped, ddof=1) / np.var(Y, ddof=1))
    theoretical_reductions.append(rho**2)

plt.plot(correlations, theoretical_reductions, 'r--', linewidth=2, label='Theoretical ($\\rho^2$)')
plt.plot(correlations, empirical_reductions, 'bo-', markersize=5, label='Empirical')
plt.xlabel('Correlation between X (pre) and Y (experiment)')
plt.ylabel('Variance Reduction Fraction')
plt.title('CUPED Variance Reduction vs Pre-Post Correlation')
plt.legend()
plt.grid(alpha=0.3)

# Annotate key points
plt.annotate('rho=0.5 -> 25% reduction', xy=(0.5, 0.25), xytext=(0.2, 0.45),
             arrowprops=dict(arrowstyle='->', color='gray'), fontsize=10)
plt.annotate('rho=0.7 -> 49% reduction', xy=(0.7, 0.49), xytext=(0.4, 0.7),
             arrowprops=dict(arrowstyle='->', color='gray'), fontsize=10)
plt.annotate('rho=0.9 -> 81% reduction', xy=(0.9, 0.81), xytext=(0.6, 0.9),
             arrowprops=dict(arrowstyle='->', color='gray'), fontsize=10)

plt.show()

print("Practical benchmarks:")
print(f"  rho=0.5 -> {0.5**2*100:.0f}% variance reduction (equivalent to 1.3x sample size)")
print(f"  rho=0.7 -> {0.7**2*100:.0f}% variance reduction (equivalent to 2.0x sample size)")
print(f"  rho=0.9 -> {0.9**2*100:.0f}% variance reduction (equivalent to 5.3x sample size)")

---

## 4. Full Implementation: `calculate_metric_cuped`

Now let’s build a production-style implementation that works with a DataFrame of user-level events across time periods.

In [ ]:
def calculate_metric_cuped(
    df: pd.DataFrame,
    value_name: str,
    user_id_name: str,
    list_user_id: list,
    date_name: str,
    periods: dict,
    metric_name: str,
    theta: float = None
) -> pd.DataFrame:
    """
    Calculate CUPED-adjusted metric for a set of users.
    
    Parameters
    ----------
    df : DataFrame with columns [user_id_name, date_name, value_name]
    value_name : name of the column containing metric values
    user_id_name : name of the column containing user identifiers
    list_user_id : list of user IDs to include in the calculation
    date_name : name of the column containing dates
    periods : dict with keys 'prepilot' and 'pilot', each containing
              a tuple (start_date, end_date)
    metric_name : name for the output metric column
    theta : float, optional
        Pre-computed theta (Cov(Y,X)/Var(X)) from pooled data.
        If None, theta is computed from the users in list_user_id.
        IMPORTANT: For valid A/B test results, pass the SAME theta
        (computed from ALL users) to both control and treatment calls.
    
    Returns
    -------
    DataFrame with columns [user_id_name, metric_name] containing
    the CUPED-adjusted metric per user.
    
    Process
    -------
    1. Compute metric for each user in the prepilot period
    2. Compute metric for each user in the pilot period
    3. Use provided theta, or calculate theta = Cov(pilot, prepilot) / Var(prepilot)
    4. Return Y_cuped = Y_pilot - theta * (Y_prepilot - mean(Y_prepilot))
    """
    # Filter to target users
    df_filtered = df[df[user_id_name].isin(list_user_id)].copy()
    df_filtered[date_name] = pd.to_datetime(df_filtered[date_name])
    
    # Step 1: Compute prepilot metric per user
    prepilot_start, prepilot_end = pd.to_datetime(periods['prepilot'][0]), pd.to_datetime(periods['prepilot'][1])
    mask_prepilot = (df_filtered[date_name] >= prepilot_start) & (df_filtered[date_name] <= prepilot_end)
    prepilot_metric = (
        df_filtered[mask_prepilot]
        .groupby(user_id_name)[value_name]
        .sum()
        .reindex(list_user_id, fill_value=0)
    )
    
    # Step 2: Compute pilot metric per user
    pilot_start, pilot_end = pd.to_datetime(periods['pilot'][0]), pd.to_datetime(periods['pilot'][1])
    mask_pilot = (df_filtered[date_name] >= pilot_start) & (df_filtered[date_name] <= pilot_end)
    pilot_metric = (
        df_filtered[mask_pilot]
        .groupby(user_id_name)[value_name]
        .sum()
        .reindex(list_user_id, fill_value=0)
    )
    
    # Step 3: Calculate theta (or use provided value)
    Y_pilot = pilot_metric.values.astype(float)
    Y_prepilot = prepilot_metric.values.astype(float)
    
    if theta is None:
        covariance = np.cov(Y_pilot, Y_prepilot, ddof=1)[0, 1]
        variance_prepilot = np.var(Y_prepilot, ddof=1)
        theta = covariance / variance_prepilot if variance_prepilot > 0 else 0
    
    # Step 4: CUPED adjustment (mean-centered covariate)
    Y_cuped = Y_pilot - theta * (Y_prepilot - Y_prepilot.mean())
    
    result = pd.DataFrame({
        user_id_name: list_user_id,
        metric_name: Y_cuped
    })
    
    return result

### QuickCart Example: Purchase Total A/B Test

QuickCart is testing a new recommendation widget. The primary metric is total purchase amount per user during the experiment. We’ll use last month’s purchase total as the CUPED covariate.

In [ ]:
np.random.seed(123)
n_users = 4000
n_days_prepilot = 30
n_days_pilot = 14

user_ids = [f"user_{i:05d}" for i in range(n_users)]

# Assign groups: first half control, second half treatment
control_users = user_ids[:n_users // 2]
treatment_users = user_ids[n_users // 2:]

# Each user has a spending propensity (log-normal -> realistic heavy tail)
user_propensity = np.random.lognormal(mean=3.0, sigma=1.0, size=n_users)

# Generate daily purchase events
records = []
base_date = pd.Timestamp('2024-01-01')

for i, uid in enumerate(user_ids):
    propensity = user_propensity[i]
    
    # Prepilot period: 30 days before experiment
    for day in range(n_days_prepilot):
        # Purchase probability depends on propensity
        if np.random.random() < 0.3:  # 30% chance of purchase on any day
            amount = np.random.exponential(scale=propensity)
            records.append({
                'user_id': uid,
                'date': base_date + pd.Timedelta(days=day),
                'purchase_amount': round(amount, 2)
            })
    
    # Pilot period: 14 days of experiment
    is_treatment = uid in treatment_users
    for day in range(n_days_pilot):
        if np.random.random() < 0.3:
            amount = np.random.exponential(scale=propensity)
            # Treatment effect: 5% increase in purchase amount
            if is_treatment:
                amount *= 1.05
            records.append({
                'user_id': uid,
                'date': base_date + pd.Timedelta(days=n_days_prepilot + day),
                'purchase_amount': round(amount, 2)
            })

df_purchases = pd.DataFrame(records)
print(f"Generated {len(df_purchases):,} purchase events for {n_users} users")
print(f"\nDate range: {df_purchases['date'].min()} to {df_purchases['date'].max()}")
print(f"\nSample records:")
df_purchases.head(10)

In [ ]:
# Define periods
periods = {
    'prepilot': ('2024-01-01', '2024-01-30'),
    'pilot': ('2024-01-31', '2024-02-13')
}

# Step 1: Compute pooled theta from ALL users (both groups combined)
# This ensures unbiased comparison — same theta used for control and treatment
all_users = control_users + treatment_users
df_purchases['date'] = pd.to_datetime(df_purchases['date'])

prepilot_all = (
    df_purchases[(df_purchases['date'] >= '2024-01-01') & (df_purchases['date'] <= '2024-01-30')]
    .groupby('user_id')['purchase_amount'].sum()
    .reindex(all_users, fill_value=0)
)
pilot_all = (
    df_purchases[(df_purchases['date'] >= '2024-01-31') & (df_purchases['date'] <= '2024-02-13')]
    .groupby('user_id')['purchase_amount'].sum()
    .reindex(all_users, fill_value=0)
)

cov_pooled = np.cov(pilot_all.values, prepilot_all.values, ddof=1)[0, 1]
var_prepilot_pooled = np.var(prepilot_all.values, ddof=1)
theta_pooled = cov_pooled / var_prepilot_pooled if var_prepilot_pooled > 0 else 0
print(f"Pooled theta (from all {len(all_users)} users): {theta_pooled:.4f}")

# Step 2: Apply CUPED to each group using the SAME pooled theta
cuped_control = calculate_metric_cuped(
    df=df_purchases,
    value_name='purchase_amount',
    user_id_name='user_id',
    list_user_id=control_users,
    date_name='date',
    periods=periods,
    metric_name='purchase_cuped',
    theta=theta_pooled
)

cuped_treatment = calculate_metric_cuped(
    df=df_purchases,
    value_name='purchase_amount',
    user_id_name='user_id',
    list_user_id=treatment_users,
    date_name='date',
    periods=periods,
    metric_name='purchase_cuped',
    theta=theta_pooled
)

# Also compute raw pilot metric for comparison
raw_control = pilot_all.loc[control_users]
raw_treatment = pilot_all.loc[treatment_users]

# Compare
print("\n" + "=" * 60)
print("RESULTS: Purchase Total A/B Test")
print("=" * 60)
print(f"\n{'Metric':<25} {'Control Mean':>12} {'Treatment Mean':>14} {'Diff':>8}")
print("-" * 60)
print(f"{'Raw Pilot Metric':<25} {raw_control.mean():>12.2f} {raw_treatment.mean():>14.2f} {raw_treatment.mean()-raw_control.mean():>8.2f}")
print(f"{'CUPED-Adjusted':<25} {cuped_control['purchase_cuped'].mean():>12.2f} {cuped_treatment['purchase_cuped'].mean():>14.2f} {cuped_treatment['purchase_cuped'].mean()-cuped_control['purchase_cuped'].mean():>8.2f}")

print(f"\n{'Metric':<25} {'Variance':>12} {'Std Dev':>10} {'p-value':>10}")
print("-" * 60)

var_raw = (raw_control.var() + raw_treatment.var()) / 2
var_adj = (cuped_control['purchase_cuped'].var() + cuped_treatment['purchase_cuped'].var()) / 2

_, p_raw = stats.ttest_ind(raw_treatment.values, raw_control.values)
_, p_cuped = stats.ttest_ind(cuped_treatment['purchase_cuped'].values, cuped_control['purchase_cuped'].values)

print(f"{'Raw Pilot Metric':<25} {var_raw:>12.1f} {np.sqrt(var_raw):>10.2f} {p_raw:>10.4f}")
print(f"{'CUPED-Adjusted':<25} {var_adj:>12.1f} {np.sqrt(var_adj):>10.2f} {p_cuped:>10.4f}")
print(f"\nVariance reduction: {(1 - var_adj/var_raw)*100:.1f}%")

---

## 5. CUPED + ML: Using Model Predictions as Covariates

### The Key Insight

Standard CUPED uses the same metric from the pre-period as the covariate. But the correlation between $X$ and $Y$ determines how much variance we reduce. If we can build a **better predictor** of the experiment-period metric using multiple pre-period features, we get a higher effective correlation and more variance reduction.

The approach:
1. Collect multiple features from the pre-period (visit count, purchase frequency, categories browsed, etc.)
2. Train a model (Ridge, LightGBM) to predict the pilot-period metric
3. Use the model’s prediction $\hat{Y}$ as the covariate $X$ in the CUPED formula

**Critical requirement**: All features must be from BEFORE the experiment. The model is trained on pre-period data only.

:::{admonition} Why this is unbiased (click to expand)
:class: dropdown

The CUPED adjustment $Y_{\text{cuped}} = Y - \theta(X - \bar{X})$ preserves the treatment effect whenever $\mathbb{E}[X | \text{treatment}] = \mathbb{E}[X | \text{control}]$.

When $X = \hat{Y} = f(\text{pre-period features})$, and randomization is done correctly, the pre-period features have the same distribution in both groups. Therefore $f$ maps them to predictions with the same expected value in both groups.

The only requirement is that the features and the model are determined **before** or **independently of** the experiment assignment. Training on pre-period data from both groups pooled is fine, since the treatment hasn’t happened yet.
:::

In [ ]:
np.random.seed(42)
n_users = 6000

# Generate rich pre-period features for each user
user_propensity = np.random.lognormal(mean=2.5, sigma=1.2, size=n_users)
user_frequency = np.random.poisson(lam=5, size=n_users)  # visits per week
user_recency = np.random.exponential(scale=7, size=n_users)  # days since last visit

# Pre-period features (observable)
features = pd.DataFrame({
    'user_id': range(n_users),
    'prepilot_purchase_total': user_propensity * np.random.exponential(3, n_users) + np.random.normal(0, 5, n_users),
    'prepilot_visit_count': user_frequency * 4 + np.random.poisson(2, n_users),
    'prepilot_days_since_last_purchase': user_recency + np.random.exponential(2, n_users),
    'prepilot_avg_session_duration': 5 + user_propensity * 0.1 + np.random.normal(0, 2, n_users),
    'prepilot_categories_browsed': np.random.poisson(lam=3 + user_frequency * 0.5, size=n_users),
    'prepilot_cart_abandonment_rate': np.clip(0.5 - user_propensity * 0.005 + np.random.normal(0, 0.15, n_users), 0, 1),
})

# Make features non-negative where needed
features['prepilot_purchase_total'] = np.maximum(features['prepilot_purchase_total'], 0)
features['prepilot_avg_session_duration'] = np.maximum(features['prepilot_avg_session_duration'], 0.5)

# Pilot-period metric: depends on propensity + noise + treatment effect
treatment_mask = np.arange(n_users) >= n_users // 2
pilot_metric = user_propensity * np.random.exponential(1.5, n_users) + np.random.normal(0, 10, n_users)
pilot_metric = np.maximum(pilot_metric, 0)
pilot_metric[treatment_mask] *= 1.03  # 3% treatment effect

features['pilot_purchase_total'] = pilot_metric

print("Feature matrix shape:", features.shape)
print("\nFeature correlations with pilot metric:")
feature_cols = [c for c in features.columns if c.startswith('prepilot_')]
for col in feature_cols:
    corr = features[col].corr(features['pilot_purchase_total'])
    print(f"  {col:<40} rho = {corr:.3f}")

In [ ]:
# Split into control and treatment
control_idx = ~treatment_mask
treatment_idx = treatment_mask

X_features = features[feature_cols].values
Y_pilot = features['pilot_purchase_total'].values

# --- Approach 1: Simple CUPED (single covariate) ---
X_simple = features['prepilot_purchase_total'].values
theta_simple = np.cov(Y_pilot, X_simple)[0, 1] / np.var(X_simple, ddof=1)
Y_cuped_simple = Y_pilot - theta_simple * (X_simple - X_simple.mean())

# --- Approach 2: Ridge regression prediction as covariate ---
ridge = Ridge(alpha=1.0)
# Use cross-validated predictions to avoid overfitting
X_pred_ridge = cross_val_predict(ridge, X_features, Y_pilot, cv=5)
theta_ridge = np.cov(Y_pilot, X_pred_ridge)[0, 1] / np.var(X_pred_ridge, ddof=1)
Y_cuped_ridge = Y_pilot - theta_ridge * (X_pred_ridge - X_pred_ridge.mean())

# --- Approach 3: LightGBM prediction as covariate ---
lgbm = lgb.LGBMRegressor(n_estimators=100, max_depth=5, learning_rate=0.1,
                          verbose=-1, random_state=42)
X_pred_lgbm = cross_val_predict(lgbm, X_features, Y_pilot, cv=5)
theta_lgbm = np.cov(Y_pilot, X_pred_lgbm)[0, 1] / np.var(X_pred_lgbm, ddof=1)
Y_cuped_lgbm = Y_pilot - theta_lgbm * (X_pred_lgbm - X_pred_lgbm.mean())

# Compare variance reductions
var_original = np.var(Y_pilot, ddof=1)
var_simple = np.var(Y_cuped_simple, ddof=1)
var_ridge = np.var(Y_cuped_ridge, ddof=1)
var_lgbm = np.var(Y_cuped_lgbm, ddof=1)

print("=" * 65)
print("CUPED VARIANCE REDUCTION: Simple vs ML Covariates")
print("=" * 65)
print(f"\n{'Method':<30} {'Variance':>10} {'Reduction':>12} {'Eff. rho':>10}")
print("-" * 65)
print(f"{'No adjustment':<30} {var_original:>10.1f} {'--':>12} {'--':>10}")
print(f"{'Simple CUPED (1 covariate)':<30} {var_simple:>10.1f} {(1-var_simple/var_original)*100:>11.1f}% {np.sqrt(1-var_simple/var_original):>10.3f}")
print(f"{'CUPED + Ridge (6 features)':<30} {var_ridge:>10.1f} {(1-var_ridge/var_original)*100:>11.1f}% {np.sqrt(1-var_ridge/var_original):>10.3f}")
print(f"{'CUPED + LightGBM (6 features)':<30} {var_lgbm:>10.1f} {(1-var_lgbm/var_original)*100:>11.1f}% {np.sqrt(1-var_lgbm/var_original):>10.3f}")

print(f"\nInterpretation: LightGBM captures non-linear patterns, giving a better")
print(f"prediction and therefore more variance reduction.")

In [ ]:
# Now run the actual hypothesis tests with each CUPED variant
methods = {
    'No adjustment': Y_pilot,
    'Simple CUPED': Y_cuped_simple,
    'CUPED + Ridge': Y_cuped_ridge,
    'CUPED + LightGBM': Y_cuped_lgbm,
}

print("\nHypothesis Test Results (true effect = 3% lift):")
print("=" * 70)
print(f"{'Method':<25} {'Control Mean':>12} {'Treatment Mean':>14} {'p-value':>10} {'Sig?':>6}")
print("-" * 70)

for name, metric in methods.items():
    ctrl = metric[control_idx]
    treat = metric[treatment_idx]
    _, p = stats.ttest_ind(treat, ctrl)
    sig = 'Yes' if p < 0.05 else 'No'
    print(f"{name:<25} {ctrl.mean():>12.2f} {treat.mean():>14.2f} {p:>10.4f} {sig:>6}")

In [ ]:
# Visualize: correlation between prediction and actual (per method)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

covariates = [
    ('Simple (prepilot total)', X_simple),
    ('Ridge prediction', X_pred_ridge),
    ('LightGBM prediction', X_pred_lgbm)
]

for ax, (name, X_cov) in zip(axes, covariates):
    rho = np.corrcoef(X_cov, Y_pilot)[0, 1]
    ax.scatter(X_cov[:500], Y_pilot[:500], alpha=0.3, s=10)
    ax.set_xlabel('Covariate (X)')
    ax.set_ylabel('Pilot Metric (Y)')
    ax.set_title(f'{name}\nrho = {rho:.3f}, reduction = {rho**2*100:.0f}%')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 6. Practical Considerations

### Rules for Using CUPED Correctly

| Rule | Why | Violation Consequence |
|------|-----|----------------------|
| Covariate must be from BEFORE the experiment | Otherwise it may be affected by treatment | Biased estimates |
| Use the same theta for both groups | Theta is estimated from pooled data | Ensures unbiased comparison |
| Works on user-level metrics | Need paired pre/post data per user | Cannot apply to session-level |
| Covariate should not be affected by treatment | Any post-randomization variable is suspect | Introduces selection bias |
| Mean difference is preserved | CUPED only reduces variance | Effect estimate unchanged |

### When CUPED Helps Most

- Metrics with high user-level persistence (purchase behavior, engagement)
- Longer pre-periods provide more stable covariate estimates
- Metrics that are NOT strongly affected by external shocks (seasonality)

### When CUPED Helps Less

- New users (no pre-period data available)
- Metrics driven primarily by external factors (flash sales, news events)
- Very short pre-periods with noisy covariates

In [ ]:
# Demonstrate: what happens if covariate is measured DURING the experiment?
# (This is the #1 mistake people make with CUPED)

np.random.seed(99)
n = 3000

# True scenario: treatment increases engagement by 2 points
baseline = np.random.normal(50, 10, n)
treatment_assignment = np.random.binomial(1, 0.5, n).astype(bool)
true_effect = 2.0

# Correct covariate: measured BEFORE experiment
X_correct = baseline + np.random.normal(0, 5, n)

# Incorrect covariate: measured DURING experiment (contaminated by treatment)
X_wrong = baseline + np.random.normal(0, 5, n) + treatment_assignment * true_effect * 0.5

# Outcome
Y = baseline + np.random.normal(0, 5, n) + treatment_assignment * true_effect

# Apply CUPED with correct vs incorrect covariate
def apply_cuped(Y, X, treatment_mask):
    theta = np.cov(Y, X)[0, 1] / np.var(X, ddof=1)
    Y_adj = Y - theta * (X - X.mean())
    ctrl = Y_adj[~treatment_mask]
    treat = Y_adj[treatment_mask]
    return treat.mean() - ctrl.mean()

# Raw difference
raw_effect = Y[treatment_assignment].mean() - Y[~treatment_assignment].mean()
correct_effect = apply_cuped(Y, X_correct, treatment_assignment)
wrong_effect = apply_cuped(Y, X_wrong, treatment_assignment)

print("Effect Estimates (true effect = 2.0):")
print(f"  Raw difference:              {raw_effect:.3f}")
print(f"  CUPED (pre-period covariate): {correct_effect:.3f}  <-- correct")
print(f"  CUPED (during-exp covariate): {wrong_effect:.3f}  <-- BIASED (attenuated)")
print(f"\nUsing a contaminated covariate attenuates the effect estimate toward zero!")
print(f"This is because the covariate already partially captures the treatment effect,")
print(f"and CUPED 'subtracts it out'.")

---

## 7. Power Simulation: CUPED Reduces Required Sample Size

Let’s run a simulation showing that CUPED allows you to detect the same effect with fewer samples (or detect smaller effects with the same sample).

In [ ]:
np.random.seed(42)
n_simulations = 2000
sample_sizes = [200, 500, 1000, 2000, 3000, 5000]
true_effect = 0.5
rho = 0.7  # correlation between pre and post

power_raw = []
power_cuped = []

for n in sample_sizes:
    rejections_raw = 0
    rejections_cuped = 0
    
    for _ in range(n_simulations):
        # Generate correlated pre/post
        cov_matrix = [[1, rho], [rho, 1]]
        data = np.random.multivariate_normal([0, 0], cov_matrix, size=n)
        X_pre = data[:, 0]
        Y_post = data[:, 1]
        
        # Add treatment effect to second half
        Y_post[n//2:] += true_effect
        
        # Raw t-test
        _, p_raw = stats.ttest_ind(Y_post[n//2:], Y_post[:n//2])
        if p_raw < 0.05:
            rejections_raw += 1
        
        # CUPED
        theta = np.cov(Y_post, X_pre)[0, 1] / np.var(X_pre, ddof=1)
        Y_adj = Y_post - theta * (X_pre - X_pre.mean())
        _, p_cuped = stats.ttest_ind(Y_adj[n//2:], Y_adj[:n//2])
        if p_cuped < 0.05:
            rejections_cuped += 1
    
    power_raw.append(rejections_raw / n_simulations)
    power_cuped.append(rejections_cuped / n_simulations)

plt.plot(sample_sizes, power_raw, 'o-', label='Raw metric', linewidth=2)
plt.plot(sample_sizes, power_cuped, 's-', label=f'CUPED (rho={rho})', linewidth=2)
plt.axhline(0.8, color='red', linestyle='--', alpha=0.5, label='80% power target')
plt.xlabel('Sample Size (per group)')
plt.ylabel('Power (proportion of significant results)')
plt.title(f'Statistical Power: Raw vs CUPED (effect={true_effect}, rho={rho})')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Find sample size needed for 80% power
for i, n in enumerate(sample_sizes):
    if power_raw[i] >= 0.8:
        print(f"Raw: ~{n} users per group needed for 80% power")
        break

for i, n in enumerate(sample_sizes):
    if power_cuped[i] >= 0.8:
        print(f"CUPED: ~{n} users per group needed for 80% power")
        break

print(f"\nWith rho={rho}, CUPED reduces required sample size by ~{(1-0.7**2)*100:.0f}%")

---

## 8. Validating CUPED: A/A Test

Before using CUPED in production, we must verify it preserves the correct Type I error rate. Under the null (no treatment effect), CUPED-adjusted p-values should be uniformly distributed.

In [ ]:
np.random.seed(42)
n_simulations = 5000
n = 2000
rho = 0.7

pvals_raw = []
pvals_cuped = []

for _ in range(n_simulations):
    # No treatment effect (A/A test)
    cov_matrix = [[1, rho], [rho, 1]]
    data = np.random.multivariate_normal([0, 0], cov_matrix, size=n)
    X_pre = data[:, 0]
    Y_post = data[:, 1]
    
    # Split randomly into two groups
    _, p_raw = stats.ttest_ind(Y_post[n//2:], Y_post[:n//2])
    pvals_raw.append(p_raw)
    
    # CUPED
    theta = np.cov(Y_post, X_pre)[0, 1] / np.var(X_pre, ddof=1)
    Y_adj = Y_post - theta * (X_pre - X_pre.mean())
    _, p_cuped = stats.ttest_ind(Y_adj[n//2:], Y_adj[:n//2])
    pvals_cuped.append(p_cuped)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(pvals_raw, bins=50, density=True, alpha=0.7, color='steelblue')
axes[0].axhline(1, color='red', linestyle='--', label='Uniform (expected)')
axes[0].set_title(f'A/A Test p-values: Raw Metric\nFPR = {np.mean(np.array(pvals_raw) < 0.05):.3f}')
axes[0].set_xlabel('p-value')
axes[0].legend()

axes[1].hist(pvals_cuped, bins=50, density=True, alpha=0.7, color='darkorange')
axes[1].axhline(1, color='red', linestyle='--', label='Uniform (expected)')
axes[1].set_title(f'A/A Test p-values: CUPED\nFPR = {np.mean(np.array(pvals_cuped) < 0.05):.3f}')
axes[1].set_xlabel('p-value')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Both distributions should be uniform (flat histogram).")
print(f"False positive rates should be close to 0.05:")
print(f"  Raw:   {np.mean(np.array(pvals_raw) < 0.05):.3f}")
print(f"  CUPED: {np.mean(np.array(pvals_cuped) < 0.05):.3f}")
print(f"\nCUPED is correctly calibrated -- it reduces variance without inflating Type I error.")

---

## Summary

| Concept | Key Takeaway |
|---------|-------------|
| CUPED formula | $Y_{\text{cuped}} = Y - \theta(X - \bar{X})$, where $\theta = \text{Cov}(Y,X)/\text{Var}(X)$ |
| Variance reduction | Factor of $(1 - \rho^2)$; rho=0.7 gives 51% reduction |
| Equivalent sample size | Variance reduction of $k$% is like having $1/(1-k)$ times more users |
| Critical requirement | Covariate $X$ must be from BEFORE the experiment |
| Effect on mean | Mean difference (treatment effect) is unchanged |
| CUPED + ML | Use model predictions as covariate; better models give more reduction |
| ML covariate | Train on pre-period features, predict pilot metric; use prediction as $X$ |
| Best when | Pre/post correlation is high (same metric, stable user behavior) |
| Validation | Always run A/A tests to confirm correct calibration |

### What’s Next

In **Week 6**, we’ll tackle ratio metrics (like revenue per session) — metrics that can’t be directly averaged per user because users contribute different numbers of observations. We’ll learn linearization, which transforms ratio metrics into CUPED-compatible form for maximum variance reduction.